# Coral Detection — Validation and Final Test Comparison

Evaluate the same three frozen checkpoints on both validation and held-out test images.

**Models** — YOLOv8s, Faster R-CNN (ResNet50-FPN), and EfficientDet-D0.

Report overall mAP50 and mAP50–95 separately for each split, together with per-class AP,
measured inference speed, and checkpoint size. Overall mAP includes Hard Coral, Soft Coral,
and quadrat_area. Coral-only mean AP is also reported to show performance without quadrat_area.

Use validation for checkpoint and parameter selection. Enable final test evaluation only after
those decisions are frozen. Keep the same checkpoints and inference settings across both splits.
Validation and test scores may differ. The report shows their measured gap in percentage points.
It does not combine them into one accuracy score or adjust them to agree.


## Evaluation metrics & mathematical formulas

Object detection is **not** scored with plain classification "accuracy," because each image contains many objects that must be both **located** and **classified**. The field-standard metric is **mean Average Precision (mAP)**. It is built up from the following definitions.

**1. Intersection over Union (IoU)** — how well a predicted box $B_p$ overlaps a ground-truth box $B_{gt}$:

$$\mathrm{IoU} = \frac{\text{area}(B_p \cap B_{gt})}{\text{area}(B_p \cup B_{gt})}$$

A prediction is a **True Positive (TP)** if it has the correct class **and** $\mathrm{IoU} \ge t$ (threshold, e.g. 0.5). Otherwise it is a **False Positive (FP)**. A ground-truth object with no matching prediction is a **False Negative (FN)**.

**2. Precision and Recall:**

$$\text{Precision} = \frac{TP}{TP + FP} \qquad\qquad \text{Recall} = \frac{TP}{TP + FN}$$

**3. F1-score** (harmonic mean of precision and recall):

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**4. Average Precision (AP)** — the area under the precision–recall curve for one class, i.e. precision $p(r)$ integrated over recall $r$:

$$\text{AP} = \int_{0}^{1} p(r)\,dr \;\approx\; \sum_{n}\left(r_{n+1}-r_{n}\right)\,p_{\text{interp}}(r_{n+1})$$

**5. mean Average Precision (mAP)** — AP averaged over all $C$ classes:

$$\text{mAP} = \frac{1}{C}\sum_{i=1}^{C}\text{AP}_i$$

Two conventions are reported (as in the COCO benchmark):
- **mAP@0.5** — TP threshold fixed at $\mathrm{IoU}=0.5$ (the "PASCAL VOC" convention).
- **mAP@0.5:0.95** — averaged over ten IoU thresholds $0.50, 0.55, \dots, 0.95$; the stricter, primary COCO metric.

**Speed** is reported as inference latency (ms/image) and throughput $\text{FPS} = 1000 / \text{latency(ms)}$.

> **Reading the results:** the `quadrat_area` box is large and easy to detect, so it earns a very high AP and pulls the overall mAP up for every model. When you argue the "best" algorithm to the panel, compare the **per-class AP for Hard Coral and Soft Coral** — that reflects the genuinely hard task — alongside the overall mAP, speed, and size trade-off.

## Setup: Install Dependencies

In [ ]:
!pip install -q ultralytics effdet timm torchmetrics pycocotools roboflow pyyaml tqdm matplotlib pandas

## Configure the reviewed dataset and checkpoint storage

Fill in the Roboflow workspace, project, version, and three checkpoint filenames in the next cell.
Use the exact dataset version and split assignment used by all training runs. `FINAL_TEST = True` enables both validation and test for this final report.
Use it after checkpoints and inference settings are frozen. Set it to `False` if you are still
choosing settings, so only validation is evaluated. This notebook performs evaluation only.


In [ ]:
MODEL_NAME="comparison"
from pathlib import Path
from getpass import getpass
from datetime import datetime
import os, json, random, hashlib, importlib.metadata
import numpy as np
import torch
from roboflow import Roboflow

# Fill these from the newly generated Roboflow version. Never paste a key here.
WORKSPACE = ""
PROJECT = ""
DATASET_VERSION = None
EXPECTED_CLASSES = {"Hard Coral", "Soft Coral", "quadrat_area"}
SEED = 42
USE_DRIVE = True
FINAL_TEST = True  # Final report on both splits. Set False during model tuning.
# Replace these with the exact filenames saved by each training notebook.
YOLO_CHECKPOINT = 'yolov8s_validator_v3_best.pt'
FRCNN_CHECKPOINT = 'faster_rcnn_validator_v3_best.pt'
EDET_CHECKPOINT = 'efficientdet_validator_v3_best.pt' 
assert WORKSPACE and PROJECT and isinstance(DATASET_VERSION, int), "Enter workspace, project and version first."
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/CoralSense')
else:
    OUTPUT_ROOT = Path.cwd() / 'coralsense_outputs'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / 'runs' / (datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '-' + MODEL_NAME)
RUN_DIR.mkdir(parents=True)
ROBOFLOW_API_KEY = getpass("Roboflow API key: ")
dataset = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(WORKSPACE).project(PROJECT).version(DATASET_VERSION).download('yolov8')
del ROBOFLOW_API_KEY
dataset_path = Path(dataset.location)

## Audit the dataset and load all three matching checkpoints

In [ ]:
import yaml
from PIL import Image, ImageOps
import torchvision.transforms.functional as TF

with (dataset_path / 'data.yaml').open() as handle:
    data = yaml.safe_load(handle)
names = data['names']
class_names = [names[k] for k in sorted(names, key=int)] if isinstance(names, dict) else list(names)
assert len(class_names) == 3 and set(class_names) == EXPECTED_CLASSES, f"Unexpected classes: {class_names}"
data['names'] = class_names
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
def image_paths(directory):
    return sorted(p for p in Path(directory).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)

def find_split(dp, split_names):
    for name in split_names:
        aliases = ['valid', 'val'] if name in ('valid', 'val') else [name]
        for split in aliases:
            for images, labels in [(dp / split / 'images', dp / split / 'labels'),
                                   (dp / 'images' / split, dp / 'labels' / split)]:
                if images.is_dir() and labels.is_dir() and image_paths(images): return images, labels
    raise FileNotFoundError(f"Missing/empty split: {split_names}. Evaluation requires the requested split.")

def read_annotations(path, label_dir, width, height, offset=0):
    boxes, labels = [], []
    label_path = Path(label_dir) / (Path(path).stem + '.txt')
    if not label_path.exists():
        raise FileNotFoundError(f"Missing label: {label_path.name}. Use an empty file only for reviewed negative images.")
    for line in label_path.read_text().splitlines():
        if not line.strip(): continue
        fields = line.split()
        if len(fields) != 5: raise ValueError(f"Expected detection boxes, not polygons: {label_path.name}")
        cid = int(fields[0]); cx, cy, w, h = map(float, fields[1:])
        if not (0 <= cid < len(class_names) and all(np.isfinite([cx,cy,w,h])) and
                0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError(f"Invalid class/box in {label_path.name}")
        box = [max(0., (cx-w/2)*width), max(0., (cy-h/2)*height),
               min(float(width), (cx+w/2)*width), min(float(height), (cy+h/2)*height)]
        if box[2] <= box[0] or box[3] <= box[1]: raise ValueError(f"Degenerate box: {label_path.name}")
        boxes.append(box); labels.append(cid + offset)
    return {'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1,4),
            'labels': torch.tensor(labels, dtype=torch.int64)}

split_dirs = {s: find_split(dataset_path, [s]) for s in ('train','valid','test')}
manifest = {}; seen_hashes = {}; fingerprint = hashlib.sha256()
for split, (images_dir, labels_dir) in split_dirs.items():
    counts = np.zeros(len(class_names), dtype=int); rows = []
    for path in image_paths(images_dir):
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen_hashes and seen_hashes[digest] != split:
            raise ValueError('Identical image found across splits: ' + path.name)
        seen_hashes[digest] = split
        with Image.open(path) as img: target = read_annotations(path, labels_dir, *img.size)
        for cid in target['labels'].tolist(): counts[cid] += 1
        label_hash = hashlib.sha256((labels_dir / (path.stem+'.txt')).read_bytes()).hexdigest()
        rows.append({'image_sha256': digest, 'label_sha256': label_hash})
    if any(counts == 0): raise ValueError(f"{split} has no examples of: {[class_names[i] for i in np.where(counts==0)[0]]}")
    # Filenames differ across exports; content-based fingerprints remain comparable.
    rows.sort(key=lambda r: (r['image_sha256'],r['label_sha256']))
    manifest[split] = {'images':len(rows),'instances':counts.tolist(),'content':rows}
    print(split, 'images:',len(rows), 'instances:',dict(zip(class_names,counts.tolist())))
fingerprint.update(json.dumps({'classes':class_names,'splits':manifest},sort_keys=True).encode())
DATASET_FINGERPRINT = fingerprint.hexdigest()
META = {'workspace':WORKSPACE,'project':PROJECT,'version':DATASET_VERSION,
        'class_names':class_names,'dataset_fingerprint':DATASET_FINGERPRINT,'seed':SEED,
        'model':MODEL_NAME,'hardware':torch.cuda.get_device_name(0) if device.type=='cuda' else 'cpu',
        'augmentation':'Roboflow version transformations only; additional random training augmentation disabled'}
(RUN_DIR/'dataset_manifest.json').write_text(json.dumps(manifest,indent=2))
(RUN_DIR/'run_metadata.json').write_text(json.dumps(META,indent=2))
versions = {}
for name in ('torch','torchvision','roboflow','numpy','Pillow','ultralytics','torchmetrics','effdet','timm','pycocotools'):
    try: versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: pass
(RUN_DIR/'package_versions.json').write_text(json.dumps(versions,indent=2))
print('Classes:',class_names,'Output:',RUN_DIR)
print('Exact duplicates checked. Related photos must also have been grouped by survey/session before export.')

def letterbox(img, size):
    w,h=img.size; ratio=min(size/w,size/h)
    nw,nh=max(1,round(w*ratio)),max(1,round(h*ratio))
    left,top=(size-nw)//2,(size-nh)//2
    canvas=Image.new('RGB',(size,size),(114,114,114))
    canvas.paste(img.resize((nw,nh),Image.Resampling.BILINEAR),(left,top))
    return canvas,(nw/w,nh/h,left,top)

from ultralytics import YOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from effdet import get_efficientdet_config, EfficientDet, DetBenchPredict
from effdet.efficientdet import HeadNet
def check_meta(meta):
    assert meta['class_names']==class_names,'Checkpoint class order differs from dataset'
    assert meta['dataset_fingerprint']==DATASET_FINGERPRINT,'Checkpoint trained on different dataset/splits'
    assert (meta['workspace'],meta['project'],meta['version'])==(WORKSPACE,PROJECT,DATASET_VERSION),'Dataset version mismatch'
yolo_path = CHECKPOINT_DIR / YOLO_CHECKPOINT
frcnn_path = CHECKPOINT_DIR / FRCNN_CHECKPOINT
edet_path = CHECKPOINT_DIR / EDET_CHECKPOINT
for p in (yolo_path,frcnn_path,edet_path):assert p.is_file(),f'Missing trained checkpoint: {p.name}'
check_meta(json.loads(yolo_path.with_suffix('.metadata.json').read_text()))
yolo_model=YOLO(str(yolo_path))
assert [yolo_model.names[i] for i in sorted(yolo_model.names)]==class_names
# Only load checkpoints you trained/trust. These contain optimizer metadata.
ckpt=torch.load(frcnn_path,map_location='cpu',weights_only=False);check_meta(ckpt['metadata'])
frcnn_model=fasterrcnn_resnet50_fpn(weights=None,weights_backbone=None,box_score_thresh=.001,box_detections_per_img=100)
frcnn_model.roi_heads.box_predictor=FastRCNNPredictor(frcnn_model.roi_heads.box_predictor.cls_score.in_features,ckpt['num_classes'])
frcnn_model.load_state_dict(ckpt['model_state_dict']);frcnn_model.to(device).eval()
eckpt=torch.load(edet_path,map_location='cpu',weights_only=False);check_meta(eckpt['metadata'])
EDET_IMG=eckpt['image_size'];config=get_efficientdet_config(eckpt['architecture'])
config.num_classes=eckpt['num_classes'];config.image_size=(EDET_IMG,EDET_IMG)
enet=EfficientDet(config,pretrained_backbone=False);enet.class_net=HeadNet(config,num_outputs=eckpt['num_classes'])
enet.load_state_dict(eckpt['model_state_dict']);edet_bench=DetBenchPredict(enet).to(device).eval()
test_img_dir,test_lbl_dir=split_dirs[EVAL_SPLIT];eval_split=EVAL_SPLIT
test_images=image_paths(test_img_dir)
yolo_size=yolo_path.stat().st_size/1024**2;frcnn_size=frcnn_path.stat().st_size/1024**2;edet_size=edet_path.stat().st_size/1024**2
print('All checkpoints match. Evaluation image counts:', {split: len(paths) for split, paths in evaluation_images.items()})


In [ ]:
## Dataset Class Distribution

#Counts of annotated **instances (bounding boxes)** and **images** per class across the
#train / val / test splits. This answers two panel comments:

##1. *"The distribution of the images per class should be discussed in the documentation."* — the table and chart below are the figures to put in your Dataset chapter.
##2. *"Consider the distribution of datasets in the comparative analysis of the algorithm."* — reference these counts when interpreting the per-class AP results later (e.g. a class with few instances explaining a low AP).

#> Note: If Roboflow augmentation was enabled, only training counts are increased; inspect the recorded version settings — the **relative proportions between classes** are what reveal imbalance. `quadrat_area` appears about once per image, so its instance count ≈ its image count.
# ── Dataset class distribution (instances + images per class, per split) ────
# Self-contained: only needs `dataset_path`, `class_names`, and `find_split`
# from the dataset-loading cell above.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

SPLITS = ['train', 'val', 'test']
inst_counts = {s: defaultdict(int) for s in SPLITS}   # bounding boxes per class
img_counts = {s: defaultdict(int) for s in SPLITS}    # images containing the class

for s in SPLITS:
    _, lbl_dir = find_split(dataset_path, [s, 'valid'] if s == 'val' else [s])
    if lbl_dir is None:
        continue
    for image_path in image_paths(find_split(dataset_path, [s])[0]):
        lbl = lbl_dir / (image_path.stem + '.txt')
        present = set()
        for line in open(lbl):
            p = line.split()
            if len(p) < 5:
                continue
            cid = int(p[0])
            if 0 <= cid < len(class_names):
                inst_counts[s][cid] += 1
                present.add(cid)
        for cid in present:
            img_counts[s][cid] += 1

# ---- Table 1: instances (bounding boxes) per class ----
inst_rows = []
for cid, name in enumerate(class_names):
    tr, va, te = inst_counts['train'][cid], inst_counts['val'][cid], inst_counts['test'][cid]
    inst_rows.append({'Class': name, 'Train': tr, 'Val': va, 'Test': te, 'Total': tr + va + te})
inst_df = pd.DataFrame(inst_rows)
total_inst = inst_df['Total'].sum()
inst_df['% of all'] = (inst_df['Total'] / total_inst * 100).round(1) if total_inst else 0

# ---- Table 2: images per class ----
img_rows = []
for cid, name in enumerate(class_names):
    tr, va, te = img_counts['train'][cid], img_counts['val'][cid], img_counts['test'][cid]
    img_rows.append({'Class': name, 'Train': tr, 'Val': va, 'Test': te, 'Total': tr + va + te})
img_df = pd.DataFrame(img_rows)

print("=" * 74)
print("CLASS DISTRIBUTION — annotated INSTANCES (bounding boxes)")
print("=" * 74)
print(inst_df.to_string(index=False))
print(f"\nTotal instances: {total_inst}")

print("\n" + "=" * 74)
print("CLASS DISTRIBUTION — IMAGES containing each class")
print("=" * 74)
print(img_df.to_string(index=False))

# ---- Imbalance ratio (over benthic classes, ignoring quadrat_area) ----
benthic_totals = inst_df[inst_df['Class'] != 'quadrat_area']['Total']
benthic_totals = benthic_totals[benthic_totals > 0]
if len(benthic_totals):
    imb = benthic_totals.max() / benthic_totals.min()
    print(f"\nBenthic class imbalance ratio (largest / smallest): {imb:.1f}×")
    print("→ Discuss this when interpreting per-class AP: rare classes tend to score lower.")

# ---- Save for the documentation ----
inst_df.to_csv(RUN_DIR / 'class_distribution_instances.csv', index=False)
img_df.to_csv(RUN_DIR / 'class_distribution_images.csv', index=False)
print("\n✓ Saved: class_distribution_instances.csv, class_distribution_images.csv")

# ---- Chart: stacked bar of instances per class, coloured by split ----
fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(class_names))
split_colors = {'Train': '#2a78d6', 'Val': '#eda100', 'Test': '#1baf7a'}
for split in ['Train', 'Val', 'Test']:
    vals = inst_df[split].values
    ax.bar(inst_df['Class'], vals, bottom=bottom, label=split,
           color=split_colors[split], edgecolor='black', linewidth=0.4)
    bottom += vals
ax.set_ylabel('Annotated instances', fontweight='bold')
ax.set_title('Dataset Class Distribution (instances per class, by split)', fontweight='bold')
ax.legend(title='Split')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(RUN_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: class_distribution.png")

## Benchmark overall mAP on validation and final test splits

Every model uses the same image lists, annotations, confidence threshold of 0.001, and maximum
of 100 detections per image. Each split gets an independent COCO metric accumulator.
Class IDs are normalized to the dataset order. Both overall metrics include all three classes.

Speed measures image decoding, preprocessing, prediction, transfers, and postprocessing using
confidence 0.25, three warm-ups, and three passes over up to five shared images. Its source split
is recorded. Checkpoint size measures the saved file, which can include optimizer state.


In [ ]:
import time
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm
import torchvision.transforms as T
to_tensor=T.ToTensor()
edet_norm=T.Normalize([.485,.456,.406],[.229,.224,.225])
@torch.inference_mode()
def predict(name,path,conf=.001):
    # All paths include image decoding, preprocessing, transfer and postprocessing.
    with Image.open(path) as original:img=original.convert('RGB')
    W,H=img.size
    if name=='YOLOv8s':
        r=yolo_model.predict(img,device=0 if device.type=='cuda' else 'cpu',conf=conf,imgsz=640,max_det=100,verbose=False)[0]
        return {'boxes':r.boxes.xyxy.cpu(),'scores':r.boxes.conf.cpu(),'labels':r.boxes.cls.cpu().long()}
    if name=='Faster R-CNN':
        p=frcnn_model([to_tensor(img).to(device)])[0];keep=p['scores']>=conf
        return {'boxes':p['boxes'][keep].cpu(),'scores':p['scores'][keep].cpu(),'labels':p['labels'][keep].cpu()-1}
    canvas,(sx,sy,left,top)=letterbox(img,EDET_IMG)
    rows=edet_bench(edet_norm(to_tensor(canvas)).unsqueeze(0).to(device))[0].cpu()
    rows=rows[rows[:,4]>=conf];boxes=rows[:,:4].clone()
    boxes[:,[0,2]]=(boxes[:,[0,2]]-left)/sx;boxes[:,[1,3]]=(boxes[:,[1,3]]-top)/sy
    boxes[:,[0,2]]=boxes[:,[0,2]].clamp(0,W);boxes[:,[1,3]]=boxes[:,[1,3]].clamp(0,H)
    keep=(boxes[:,2]>boxes[:,0])&(boxes[:,3]>boxes[:,1])
    return {'boxes':boxes[keep],'scores':rows[keep,4],'labels':rows[keep,5].long()-1}

results = {}
per_class = {}
model_sizes = {'YOLOv8s': yolo_size, 'Faster R-CNN': frcnn_size, 'EfficientDet-D0': edet_size}
for name, size in model_sizes.items():
    results[name] = {'size': size}
    per_class[name] = {}
    for split in evaluation_splits:
        metric = MeanAveragePrecision(box_format='xyxy', class_metrics=True)
        labels_dir = split_dirs[split][1]
        for path in tqdm(evaluation_images[split], desc=f'{name} {split} AP'):
            with Image.open(path) as image:
                truth = read_annotations(path, labels_dir, *image.size)
            metric.update([predict(name, path)], [truth])
        measured = metric.compute()
        per_class[name][split] = measured
        results[name][split] = {
            'images': len(evaluation_images[split]),
            'map50': float(measured['map_50']),
            'map': float(measured['map']),
        }
        print(name, split, results[name][split])

    timing_images = evaluation_images[timing_split][:5]
    for _ in range(3):
        predict(name, timing_images[0], conf=.25)
    times = []
    for _ in range(3):
        for path in timing_images:
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start = time.perf_counter()
            predict(name, path, conf=.25)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            times.append((time.perf_counter() - start) * 1000)
    ms = float(np.mean(times))
    results[name].update({'ms': ms, 'fps': 1000 / ms, 'timing_split': timing_split})
    print(name, 'latency (ms):', ms, 'size (MiB):', size)

benchmark = {
    'metadata': META,
    'splits': list(evaluation_splits),
    'final_test_enabled': FINAL_TEST,
    'evaluation_parameters': {'confidence': .001, 'max_detections': 100},
    'timing_parameters': {'split': timing_split, 'confidence': .25, 'warmups': 3,
                          'images': len(timing_images), 'repeats': 3},
    'results': results,
}
(RUN_DIR / 'benchmark.json').write_text(json.dumps(benchmark, indent=2))


## Visualize validation and test performance, speed, and size


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = list(results)
fps = [results[name]['fps'] for name in models]
sizes = [results[name]['size'] for name in models]
colors = ['#4CAF50', '#FF9800', '#2196F3']
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Coral Detection — Measured Model Comparison', fontsize=16, fontweight='bold')

for ax, values, title, ylabel in [
    (axes[0, 0], fps, f'Inference speed ({timing_split} sample)', 'FPS'),
    (axes[0, 1], sizes, 'Checkpoint file size', 'MiB'),
]:
    ax.bar(models, values, color=colors, edgecolor='black')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=.3)
    for index, value in enumerate(values):
        ax.text(index, value, f'{value:.1f}', ha='center', va='bottom')

x = np.arange(len(models))
width = .35
split_colors = {'valid': '#1c6d7d', 'test': '#eda100'}
for ax, metric_name, title in [
    (axes[1, 0], 'map50', 'Overall mAP50'),
    (axes[1, 1], 'map', 'Overall mAP50–95'),
]:
    for index, split in enumerate(evaluation_splits):
        offset = (index - (len(evaluation_splits) - 1) / 2) * width
        values = [100 * results[name][split][metric_name] for name in models]
        bars = ax.bar(x + offset, values, width, label=split, color=split_colors[split], edgecolor='black')
        ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)
    ax.set_xticks(x, models)
    ax.set_ylim(0, 110)
    ax.set_ylabel('mAP (%)')
    ax.set_title(title)
    ax.legend(title='Split')
    ax.grid(axis='y', alpha=.3)

plt.tight_layout()
plt.savefig(RUN_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## Generate the validation and final test report

Tables report mAP as percentages. Gaps use **validation minus test** in percentage points,
with absolute gaps alongside them. A positive signed gap means validation scored higher.
No closeness threshold is imposed. A small gap alone does not establish generalization or
rule out leakage. Interpret it alongside split sizes, survey/session separation, class distribution,
and per-class AP. Use validation rankings for model selection and test scores for final reporting.


In [ ]:
import pandas as pd

rows = []
class_rows = []
coral_classes = [name for name in class_names if name != 'quadrat_area']
for name in models:
    for split in evaluation_splits:
        measured = per_class[name][split]
        aps = {
            class_names[int(cid)]: float(ap)
            for cid, ap in zip(measured['classes'].tolist(), measured['map_per_class'].reshape(-1).tolist())
        }
        coral_values = [aps[label] for label in coral_classes if label in aps and aps[label] >= 0]
        coral_map = float(np.mean(coral_values)) if len(coral_values) == len(coral_classes) else None
        results[name][split]['coral_map'] = coral_map
        rows.append({
            'Model': name,
            'Split': split,
            'Images': results[name][split]['images'],
            'Overall mAP50 (%)': 100 * results[name][split]['map50'],
            'Overall mAP50–95 (%)': 100 * results[name][split]['map'],
            'Coral-only AP50–95 (%)': 100 * coral_map if coral_map is not None else None,
            'Inference (ms)': results[name]['ms'],
            'FPS': results[name]['fps'],
            'Timing split': results[name]['timing_split'],
            'Checkpoint (MiB)': results[name]['size'],
        })
        for label in class_names:
            ap = aps.get(label, -1)
            class_rows.append({'Model': name, 'Split': split, 'Class': label,
                               'AP50–95 (%)': 100 * ap if ap >= 0 else None})

df = pd.DataFrame(rows)
pc_df = pd.DataFrame(class_rows)
print('OVERALL VALIDATION AND FINAL TEST RESULTS')
print(df.round(2).to_string(index=False))
print('\nPER-CLASS AP50–95')
print(pc_df.round(2).to_string(index=False))

gap_rows = []
if FINAL_TEST:
    for name in models:
        row = {'Model': name}
        for key, label in [('map50', 'mAP50'), ('map', 'mAP50–95')]:
            gap = 100 * (results[name]['valid'][key] - results[name]['test'][key])
            row[f'{label} valid minus test (pp)'] = gap
            row[f'{label} absolute gap (pp)'] = abs(gap)
        gap_rows.append(row)
    gap_df = pd.DataFrame(gap_rows)
    print('\nVALIDATION–TEST GAP IN PERCENTAGE POINTS')
    print(gap_df.round(2).to_string(index=False))
else:
    gap_df = pd.DataFrame()
    print('\nFinal test evaluation is disabled. Freeze model selection before enabling FINAL_TEST.')

best_validation = max(models, key=lambda name: results[name]['valid']['map'])
fastest = max(models, key=lambda name: results[name]['fps'])
smallest = min(models, key=lambda name: results[name]['size'])
print('\nHighest validation mAP50–95:', best_validation)
if FINAL_TEST:
    selected = results[best_validation]['test']
    print('Final test results for the validation-selected model:', best_validation)
    print(f"mAP50: {selected['map50']:.2%}; mAP50–95: {selected['map']:.2%}")
print('Fastest on the shared timing sample:', fastest)
print('Smallest checkpoint file:', smallest)
print('Report both splits separately. Their measured values may differ.')


## Detection preview on a shared image

Use the first test image when final test evaluation is enabled, otherwise use the first validation
image. This preview illustrates model output. Quantitative results use every image in each split.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
sample=test_images[0]
fig,axes=plt.subplots(1,3,figsize=(21,7))
for ax,name in zip(axes,results):
    with Image.open(sample) as original:img=original.convert('RGB')
    prediction=predict(name,sample,conf=.3);ax.imshow(img);ax.set_title(f'{name} ({timing_split})')
    for box,score,cid in zip(prediction['boxes'],prediction['scores'],prediction['labels']):
        x1,y1,x2,y2=box.tolist()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='lime'))
        ax.text(x1,y1,f'{class_names[int(cid)]} {float(score):.2f}',color='lime')
plt.show()


## Export Results

In [ ]:
df.to_csv(RUN_DIR / 'model_comparison_report.csv', index=False)
pc_df.to_csv(RUN_DIR / 'per_class_ap.csv', index=False)
if FINAL_TEST:
    gap_df.to_csv(RUN_DIR / 'validation_test_gap.csv', index=False)
benchmark['results'] = results
(RUN_DIR / 'benchmark.json').write_text(json.dumps(benchmark, indent=2, allow_nan=False))
print('Reports, charts, dataset audit, and metadata:', RUN_DIR)
print('Evaluated splits:', ', '.join(evaluation_splits))
print('Select settings using validation. Keep final test results separate and preserve the frozen checkpoints.')
